## SAT Loggbok

tolkade news alert till Wikidata platser
* denna notebook [387_SATLoggbok.ipynb](https://github.com/salgo60/Stockholm_Archipelago_Trail/tree/main/Notebook/387_SATLoggbok.ipynb)
* Issue [#387](https://github.com/salgo60/Stockholm_Archipelago_Trail/issues/387)


In [1]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2026-06-04 17:50


In [2]:

import requests
import json
from datetime import datetime

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

query = r"""
SELECT
  ?place
  ?placeLabel
  ?coord
  ?SATpartLabel
  ?type
  ?osmNode
  ?osmWay
  ?osmRelation
  ?label_sv
  ?label_en
  ?label_de
  ?label_fr
WHERE {

  VALUES (?place ?type) {
    (wd:Q134007891 "sale_and_stamp")
    (wd:Q134676671 "sale_and_stamp")
    (wd:Q18291743  "sale_and_stamp")
    (wd:Q134006996 "sale_and_stamp")
    (wd:Q133829736 "sale_and_stamp")
    (wd:Q136337707 "sale_and_stamp")
    (wd:Q133812552 "sale_and_stamp")
    (wd:Q133812682 "sale_and_stamp")
    (wd:Q134705878 "sale_and_stamp")
    (wd:Q134571436 "sale_and_stamp")
    (wd:Q133502889 "sale_and_stamp")
    (wd:Q134572952 "sale_and_stamp")
    (wd:Q134473761 "sale_and_stamp")
    (wd:Q15695240  "sale_and_stamp")
    (wd:Q134529162 "sale_and_stamp")
    (wd:Q133840505 "sale_and_stamp")
    (wd:Q140053969 "sale_and_stamp")
    (wd:Q134589221 "sale_and_stamp")
    (wd:Q134311495 "sale_and_stamp")
    (wd:Q135437122 "sale_and_stamp")
    (wd:Q140054528 "sale_and_stamp")

    (wd:Q134523792 "stamp_only")
    (wd:Q134705637 "stamp_only")
    (wd:Q134581015 "stamp_only")
    (wd:Q134311974 "stamp_only")
    (wd:Q134589066 "stamp_only")
    (wd:Q134311548 "stamp_only")
    (wd:Q134587974 "stamp_only")
    (wd:Q134776972 "stamp_only")
    (wd:Q134306927 "stamp_only")
    (wd:Q135003463 "stamp_only")
    (wd:Q134527248 "stamp_only")
  }

  OPTIONAL {
    ?place wdt:P2789 ?SATpart .
    ?SATpart wdt:P361 wd:Q131318799 .
  }
  OPTIONAL { ?place wdt:P11693 ?osmNode }
  OPTIONAL { ?place wdt:P10689 ?osmWay }
  OPTIONAL { ?place wdt:P402 ?osmRelation }

  OPTIONAL { ?place rdfs:label ?label_sv FILTER(LANG(?label_sv)="sv") }
  OPTIONAL { ?place rdfs:label ?label_en FILTER(LANG(?label_en)="en") }
  OPTIONAL { ?place rdfs:label ?label_de FILTER(LANG(?label_de)="de") }
  OPTIONAL { ?place rdfs:label ?label_fr FILTER(LANG(?label_fr)="fr") }
  ?place wdt:P625 ?coord .

  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "sv,en".
  }
}
ORDER BY ?placeLabel
"""

response = requests.get(
    SPARQL_ENDPOINT,
    params={
        "query": query,
        "format": "json"
    },
    headers={
        "User-Agent": "SAT-Passport-Exporter/1.0 salgo60@msn.com"
    }
)

response.raise_for_status()

data = response.json()


def parse_wkt_point(wkt):
    """
    Point(18.12345 59.12345)
    -> lat, lon
    """
    wkt = wkt.replace("Point(", "").replace(")", "")
    lon, lat = map(float, wkt.split())
    return lat, lon


locations = []

for row in data["results"]["bindings"]:

    qid = row["place"]["value"].split("/")[-1]

    names = {}

    for lang in ["sv", "en", "de", "fr"]:
        key = f"label_{lang}"
    
        if key in row:
            names[lang] = row[key]["value"]
        
    lat, lon = parse_wkt_point(
        row["coord"]["value"]
    )

    osm_type = None
    osm_id = None

    if "osmNode" in row:
        osm_type = "node"
        osm_id = row["osmNode"]["value"]

    elif "osmWay" in row:
        osm_type = "way"
        osm_id = row["osmWay"]["value"]

    elif "osmRelation" in row:
        osm_type = "relation"
        osm_id = row["osmRelation"]["value"]

    location = {
        "wikidata": qid,
        "name": row["placeLabel"]["value"],  # fallback
        "names": names,
        "type": row["type"]["value"],
        "latitude": lat,
        "longitude": lon
    }
    
    if osm_type:
        location["osm"] = {
            "type": osm_type,
            "id": osm_id
        }
    
    if "SATpartLabel" in row:
        location["sat_part"] = row["SATpartLabel"]["value"]
    

    locations.append(location)

    from datetime import datetime, UTC



export = {
    "dataset": "sat-passport-locations",
    "version": "2026.1",
    "generated": datetime.now(UTC).isoformat(),
    "generated_by": "SAT Passport Notebook",
    "location_count": len(locations),
    "locations": locations
}


with open(
    "sat-passport-locations-2026.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        export,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"Created sat-passport-locations-2026.json "
    f"with {len(locations)} locations"
)

# Preview first 3 records
print(json.dumps(export["locations"][:3], indent=2, ensure_ascii=False))



Created sat-passport-locations-2026.json with 33 locations
[
  {
    "wikidata": "Q134676671",
    "name": "Arholma Nord",
    "names": {
      "sv": "Arholma Nord",
      "en": "Arholma Nord",
      "de": "Arholma Nord",
      "fr": "Arholma Nord"
    },
    "type": "sale_and_stamp",
    "latitude": 59.86079312,
    "longitude": 19.123592134,
    "osm": {
      "type": "node",
      "id": "5745003239"
    },
    "sat_part": "SAT Arholma"
  },
  {
    "wikidata": "Q134007891",
    "name": "Arholma handel",
    "names": {
      "sv": "Arholma handel",
      "en": "Arholma grocery",
      "de": "Arholma handel",
      "fr": "Arholma épicerie"
    },
    "type": "sale_and_stamp",
    "latitude": 59.851166,
    "longitude": 19.108069,
    "osm": {
      "type": "node",
      "id": "1611299267"
    },
    "sat_part": "SAT Arholma"
  },
  {
    "wikidata": "Q15695240",
    "name": "Bygdemuseet Ornö sockenstuga",
    "names": {
      "sv": "Bygdemuseet Ornö sockenstuga",
      "en": "The loca

In [3]:
end_time = time.time()
duration = end_time - start_time

print(f"Finished in {duration:.2f} seconds.")


Finished in 0.19 seconds.
